# 03 - Comandos DML em Tabelas Delta Lake

Demonstra operações DML (**INSERT**, **UPDATE**, **DELETE**) em tabelas Delta Lake armazenadas no MinIO, além de recursos como **HISTORY** e **TIME TRAVEL**.

**Pré-requisitos:** Notebook `02` executado (tabelas Delta no bucket `bronze`).

## 1. Configuração e SparkSession

In [2]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta import *
from delta.tables import DeltaTable

load_dotenv(override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

spark = (
    SparkSession.builder
    .appName('DML_Delta_Lake')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada com sucesso!')
spark

SparkSession criada com sucesso!


## 2. Registrar Tabelas Delta como SQL Tables

In [3]:
# Registrar as tabelas Delta Lake para uso com Spark SQL
tabelas_delta = ['regiao', 'estado', 'municipio', 'marca', 'modelo',
                 'cliente', 'endereco', 'telefone', 'carro', 'apolice', 'sinistro']

for tabela in tabelas_delta:
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tabela}
        USING delta
        LOCATION '{delta_path}'
    """)

# Listar tabelas registradas
print('Tabelas registradas no Spark:')
spark.sql('SHOW TABLES').show(truncate=False)

26/04/28 12:32:23 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Tabelas registradas no Spark:
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|default  |apolice  |false      |
|default  |carro    |false      |
|default  |cliente  |false      |
|default  |endereco |false      |
|default  |estado   |false      |
|default  |marca    |false      |
|default  |modelo   |false      |
|default  |municipio|false      |
|default  |regiao   |false      |
|default  |sinistro |false      |
|default  |telefone |false      |
+---------+---------+-----------+



## 3. Consultar Dados Atuais (SELECT)

In [4]:
# Visualizar as tabelas de domínio
print('=== REGIÕES ===')
spark.sql('SELECT * FROM regiao ORDER BY cd_regiao').show()

print('=== MARCAS ===')
spark.sql('SELECT * FROM marca ORDER BY cd_marca').show()

print('=== MODELOS (primeiros 10) ===')
spark.sql('SELECT * FROM modelo ORDER BY cd_modelo LIMIT 10').show()

=== REGIÕES ===


26/04/28 12:32:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+------------+
|cd_regiao|   nm_regiao|
+---------+------------+
|        1|       NORTE|
|        2|    NORDESTE|
|        3|     SUDESTE|
|        4|         SUL|
|        5|CENTRO-OESTE|
+---------+------------+

=== MARCAS ===


+--------+----------+
|cd_marca|  nm_marca|
+--------+----------+
|       1| CHEVROLET|
|       2|VOLKSWAGEN|
|       3|      FIAT|
|       4|      FORD|
|       5|    TOYOTA|
|       6|     HONDA|
|       7|   RENAULT|
|       8|   HYUNDAI|
|       9|    NISSAN|
|      10|      JEEP|
+--------+----------+

=== MODELOS (primeiros 10) ===


+---------+--------+-----------+
|cd_modelo|cd_marca|  nm_modelo|
+---------+--------+-----------+
|        1|       1|       ONIX|
|        2|       1|     PRISMA|
|        3|       1|    TRACKER|
|        4|       1|       SPIN|
|        5|       1|      CRUZE|
|        6|       1|        S10|
|        7|       1|    EQUINOX|
|        8|       1|TRAILBLAZER|
|        9|       1|    MONTANA|
|       10|       1|     CAMARO|
+---------+--------+-----------+



In [5]:
# Contagem de registros por tabela
print(f'{"Tabela":<15} {"Registros":>10}')
print('-' * 27)
for tabela in tabelas_delta:
    count = spark.sql(f'SELECT COUNT(*) as cnt FROM {tabela}').collect()[0]['cnt']
    print(f'{tabela:<15} {count:>10}')

Tabela           Registros
---------------------------


regiao                   5


estado                  27


municipio             5570
marca                   10
modelo                 100


cliente              20010


endereco             20010


telefone             20010


carro                10002


apolice              10000


sinistro             10000


---
## 4. INSERT - Inserir Novos Registros

Vamos inserir novos registros nas tabelas `marca`, `modelo` e `cliente`.

In [6]:
# INSERT - Novas marcas
print('--- INSERT: Novas marcas ---')
spark.sql("SELECT COUNT(*) as antes FROM marca").show()

spark.sql("""
    INSERT INTO marca VALUES
    (11, 'TESLA'),
    (12, 'BYD'),
    (13, 'GWM')
""")

spark.sql("SELECT * FROM marca ORDER BY cd_marca").show()
print('3 novas marcas inseridas!')

--- INSERT: Novas marcas ---
+-----+
|antes|
+-----+
|   10|
+-----+



+--------+----------+
|cd_marca|  nm_marca|
+--------+----------+
|       1| CHEVROLET|
|       2|VOLKSWAGEN|
|       3|      FIAT|
|       4|      FORD|
|       5|    TOYOTA|
|       6|     HONDA|
|       7|   RENAULT|
|       8|   HYUNDAI|
|       9|    NISSAN|
|      10|      JEEP|
|      11|     TESLA|
|      12|       BYD|
|      13|       GWM|
+--------+----------+

3 novas marcas inseridas!


In [7]:
# INSERT - Novos modelos (para as novas marcas)
print('--- INSERT: Novos modelos ---')

spark.sql("""
    INSERT INTO modelo VALUES
    (101, 11, 'MODEL 3'),
    (102, 11, 'MODEL Y'),
    (103, 12, 'DOLPHIN'),
    (104, 12, 'SONG PLUS'),
    (105, 13, 'HAVAL H6')
""")

spark.sql("SELECT * FROM modelo WHERE cd_modelo >= 101 ORDER BY cd_modelo").show()
print('5 novos modelos inseridos!')

--- INSERT: Novos modelos ---


+---------+--------+---------+
|cd_modelo|cd_marca|nm_modelo|
+---------+--------+---------+
|      101|      11|  MODEL 3|
|      102|      11|  MODEL Y|
|      103|      12|  DOLPHIN|
|      104|      12|SONG PLUS|
|      105|      13| HAVAL H6|
+---------+--------+---------+

5 novos modelos inseridos!


In [8]:
# INSERT - Novo cliente
print('--- INSERT: Novo cliente ---')

spark.sql("""
    INSERT INTO cliente VALUES
    (99001, 'JOAO DA SILVA TESTE', '12345678900', 'M', '1990-05-15'),
    (99002, 'MARIA SOUZA TESTE', '98765432100', 'F', '1985-11-20')
""")

spark.sql("SELECT * FROM cliente WHERE cd_cliente >= 99000").show()
print('2 novos clientes inseridos!')

--- INSERT: Novo cliente ---


+----------+-------------------+-----------+----+-------------+
|cd_cliente|               nome|        cpf|sexo|dt_nascimento|
+----------+-------------------+-----------+----+-------------+
|     99001|JOAO DA SILVA TESTE|12345678900|   M|   1990-05-15|
|     99002|  MARIA SOUZA TESTE|98765432100|   F|   1985-11-20|
+----------+-------------------+-----------+----+-------------+

2 novos clientes inseridos!


---
## 5. UPDATE - Atualizar Registros

Vamos atualizar registros existentes nas tabelas.

In [9]:
# UPDATE - Atualizar nome de marca
print('--- UPDATE: Renomear marca ---')
print('ANTES:')
spark.sql("SELECT * FROM marca WHERE cd_marca = 11").show()

spark.sql("""
    UPDATE marca SET nm_marca = 'TESLA MOTORS' WHERE cd_marca = 11
""")

print('DEPOIS:')
spark.sql("SELECT * FROM marca WHERE cd_marca = 11").show()

--- UPDATE: Renomear marca ---
ANTES:


+--------+--------+
|cd_marca|nm_marca|
+--------+--------+
|      11|   TESLA|
+--------+--------+

DEPOIS:


+--------+------------+
|cd_marca|    nm_marca|
+--------+------------+
|      11|TESLA MOTORS|
+--------+------------+



In [10]:
# UPDATE - Atualizar nome do cliente
print('--- UPDATE: Atualizar cliente ---')
print('ANTES:')
spark.sql("SELECT * FROM cliente WHERE cd_cliente = 99001").show()

spark.sql("""
    UPDATE cliente 
    SET nome = 'JOAO DA SILVA ATUALIZADO', 
        cpf = '11122233344'
    WHERE cd_cliente = 99001
""")

print('DEPOIS:')
spark.sql("SELECT * FROM cliente WHERE cd_cliente = 99001").show()

--- UPDATE: Atualizar cliente ---
ANTES:
+----------+-------------------+-----------+----+-------------+
|cd_cliente|               nome|        cpf|sexo|dt_nascimento|
+----------+-------------------+-----------+----+-------------+
|     99001|JOAO DA SILVA TESTE|12345678900|   M|   1990-05-15|
+----------+-------------------+-----------+----+-------------+

DEPOIS:


+----------+--------------------+-----------+----+-------------+
|cd_cliente|                nome|        cpf|sexo|dt_nascimento|
+----------+--------------------+-----------+----+-------------+
|     99001|JOAO DA SILVA ATU...|11122233344|   M|   1990-05-15|
+----------+--------------------+-----------+----+-------------+



In [11]:
# UPDATE com DeltaTable API (alternativa ao SQL)
print('--- UPDATE via DeltaTable API ---')
from pyspark.sql.functions import lit

dt_marca = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/marca')

dt_marca.update(
    condition="cd_marca = 12",
    set={"nm_marca": lit("BYD AUTO")}
)

spark.sql("SELECT * FROM marca WHERE cd_marca = 12").show()

--- UPDATE via DeltaTable API ---


+--------+--------+
|cd_marca|nm_marca|
+--------+--------+
|      12|BYD AUTO|
+--------+--------+



---
## 6. DELETE - Remover Registros

Vamos deletar registros de tabelas Delta.

In [12]:
# DELETE - Remover marca via SQL
print('--- DELETE: Remover marca GWM ---')
print('ANTES:')
spark.sql("SELECT * FROM marca ORDER BY cd_marca").show()

spark.sql("DELETE FROM marca WHERE cd_marca = 13")

print('DEPOIS:')
spark.sql("SELECT * FROM marca ORDER BY cd_marca").show()

--- DELETE: Remover marca GWM ---
ANTES:
+--------+------------+
|cd_marca|    nm_marca|
+--------+------------+
|       1|   CHEVROLET|
|       2|  VOLKSWAGEN|
|       3|        FIAT|
|       4|        FORD|
|       5|      TOYOTA|
|       6|       HONDA|
|       7|     RENAULT|
|       8|     HYUNDAI|
|       9|      NISSAN|
|      10|        JEEP|
|      11|TESLA MOTORS|
|      12|    BYD AUTO|
|      13|         GWM|
+--------+------------+

DEPOIS:


+--------+------------+
|cd_marca|    nm_marca|
+--------+------------+
|       1|   CHEVROLET|
|       2|  VOLKSWAGEN|
|       3|        FIAT|
|       4|        FORD|
|       5|      TOYOTA|
|       6|       HONDA|
|       7|     RENAULT|
|       8|     HYUNDAI|
|       9|      NISSAN|
|      10|        JEEP|
|      11|TESLA MOTORS|
|      12|    BYD AUTO|
+--------+------------+



In [13]:
# DELETE - Remover cliente via SQL
print('--- DELETE: Remover cliente teste ---')
spark.sql("SELECT * FROM cliente WHERE cd_cliente >= 99000").show()

spark.sql("DELETE FROM cliente WHERE cd_cliente = 99002")

print('Apos DELETE:')
spark.sql("SELECT * FROM cliente WHERE cd_cliente >= 99000").show()

--- DELETE: Remover cliente teste ---
+----------+--------------------+-----------+----+-------------+
|cd_cliente|                nome|        cpf|sexo|dt_nascimento|
+----------+--------------------+-----------+----+-------------+
|     99001|JOAO DA SILVA ATU...|11122233344|   M|   1990-05-15|
|     99002|   MARIA SOUZA TESTE|98765432100|   F|   1985-11-20|
+----------+--------------------+-----------+----+-------------+

Apos DELETE:


+----------+--------------------+-----------+----+-------------+
|cd_cliente|                nome|        cpf|sexo|dt_nascimento|
+----------+--------------------+-----------+----+-------------+
|     99001|JOAO DA SILVA ATU...|11122233344|   M|   1990-05-15|
+----------+--------------------+-----------+----+-------------+



In [14]:
# DELETE via DeltaTable API
print('--- DELETE via DeltaTable API ---')
dt_modelo = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/modelo')

print('Modelos >= 101 ANTES:')
spark.sql("SELECT * FROM modelo WHERE cd_modelo >= 101").show()

dt_modelo.delete("cd_modelo = 105")

print('Modelos >= 101 DEPOIS:')
spark.sql("SELECT * FROM modelo WHERE cd_modelo >= 101").show()

--- DELETE via DeltaTable API ---
Modelos >= 101 ANTES:
+---------+--------+---------+
|cd_modelo|cd_marca|nm_modelo|
+---------+--------+---------+
|      104|      12|SONG PLUS|
|      105|      13| HAVAL H6|
|      101|      11|  MODEL 3|
|      103|      12|  DOLPHIN|
|      102|      11|  MODEL Y|
+---------+--------+---------+

Modelos >= 101 DEPOIS:


+---------+--------+---------+
|cd_modelo|cd_marca|nm_modelo|
+---------+--------+---------+
|      104|      12|SONG PLUS|
|      101|      11|  MODEL 3|
|      103|      12|  DOLPHIN|
|      102|      11|  MODEL Y|
+---------+--------+---------+



---
## 7. HISTORY - Histórico de Versões Delta

O Delta Lake mantém um log de transações que permite visualizar o histórico completo de alterações.

In [15]:
# Histórico da tabela marca
print('=== HISTORICO DA TABELA MARCA ===')
dt_marca = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/marca')
dt_marca.history().select('version', 'timestamp', 'operation', 'operationMetrics').show(truncate=False)

=== HISTORICO DA TABELA MARCA ===
+-------+-------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                                                                                                            |
+-------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [16]:
# Histórico via Spark SQL
print('=== HISTORICO DA TABELA CLIENTE ===')
spark.sql('DESCRIBE HISTORY cliente').select('version', 'timestamp', 'operation').show(truncate=False)

=== HISTORICO DA TABELA CLIENTE ===
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|3      |2026-04-28 12:33:47|DELETE   |
|2      |2026-04-28 12:33:34|UPDATE   |
|1      |2026-04-28 12:33:23|WRITE    |
|0      |2026-04-28 12:30:42|WRITE    |
+-------+-------------------+---------+



---
## 8. TIME TRAVEL - Viagem no Tempo

O Delta Lake permite ler versões anteriores dos dados.

In [17]:
# Versão atual da tabela marca
print('=== MARCA - VERSAO ATUAL ===')
spark.sql('SELECT * FROM marca ORDER BY cd_marca').show()

=== MARCA - VERSAO ATUAL ===
+--------+------------+
|cd_marca|    nm_marca|
+--------+------------+
|       1|   CHEVROLET|
|       2|  VOLKSWAGEN|
|       3|        FIAT|
|       4|        FORD|
|       5|      TOYOTA|
|       6|       HONDA|
|       7|     RENAULT|
|       8|     HYUNDAI|
|       9|      NISSAN|
|      10|        JEEP|
|      11|TESLA MOTORS|
|      12|    BYD AUTO|
+--------+------------+



In [18]:
# Time Travel - Ler versão 0 (estado original antes de qualquer DML)
print('=== MARCA - VERSAO 0 (original) ===')
df_v0 = spark.read.format('delta').option('versionAsOf', 0).load(f's3a://{BRONZE_BUCKET}/marca')
df_v0.orderBy('cd_marca').show()

=== MARCA - VERSAO 0 (original) ===


+--------+----------+
|cd_marca|  nm_marca|
+--------+----------+
|       1| CHEVROLET|
|       2|VOLKSWAGEN|
|       3|      FIAT|
|       4|      FORD|
|       5|    TOYOTA|
|       6|     HONDA|
|       7|   RENAULT|
|       8|   HYUNDAI|
|       9|    NISSAN|
|      10|      JEEP|
+--------+----------+



In [19]:
# Time Travel - Comparar versão original vs atual
print('=== COMPARACAO: Versao 0 vs Atual ===')
df_original = spark.read.format('delta').option('versionAsOf', 0).load(f's3a://{BRONZE_BUCKET}/marca')
df_atual = spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/marca')

print(f'Versao 0: {df_original.count()} registros')
print(f'Atual:    {df_atual.count()} registros')

# Mostrar registros que foram adicionados (existem no atual, mas não na v0)
print('\nRegistros ADICIONADOS:')
df_atual.subtract(df_original).show()

=== COMPARACAO: Versao 0 vs Atual ===


Versao 0: 10 registros
Atual:    12 registros

Registros ADICIONADOS:


+--------+------------+
|cd_marca|    nm_marca|
+--------+------------+
|      11|TESLA MOTORS|
|      12|    BYD AUTO|
+--------+------------+



---
## 9. Resumo Final

In [20]:
print('=' * 60)
print('RESUMO DAS OPERACOES DML REALIZADAS')
print('=' * 60)
print()
print('INSERT:')
print('  - 3 novas marcas (TESLA, BYD, GWM)')
print('  - 5 novos modelos (MODEL 3, MODEL Y, DOLPHIN, etc.)')
print('  - 2 novos clientes')
print()
print('UPDATE:')
print('  - marca TESLA -> TESLA MOTORS (via SQL)')
print('  - marca BYD -> BYD AUTO (via DeltaTable API)')
print('  - cliente 99001 nome e CPF atualizados')
print()
print('DELETE:')
print('  - marca GWM removida (via SQL)')
print('  - cliente 99002 removido (via SQL)')
print('  - modelo HAVAL H6 removido (via DeltaTable API)')
print()
print('HISTORY e TIME TRAVEL:')
print('  - Historico completo de transacoes')
print('  - Leitura de versoes anteriores (versionAsOf)')
print('  - Comparacao entre versoes')
print('=' * 60)

RESUMO DAS OPERACOES DML REALIZADAS

INSERT:
  - 3 novas marcas (TESLA, BYD, GWM)
  - 5 novos modelos (MODEL 3, MODEL Y, DOLPHIN, etc.)
  - 2 novos clientes

UPDATE:
  - marca TESLA -> TESLA MOTORS (via SQL)
  - marca BYD -> BYD AUTO (via DeltaTable API)
  - cliente 99001 nome e CPF atualizados

DELETE:
  - marca GWM removida (via SQL)
  - cliente 99002 removido (via SQL)
  - modelo HAVAL H6 removido (via DeltaTable API)

HISTORY e TIME TRAVEL:
  - Historico completo de transacoes
  - Leitura de versoes anteriores (versionAsOf)
  - Comparacao entre versoes


In [21]:
spark.stop()
print('SparkSession finalizada.')

SparkSession finalizada.
